Load and inspect the data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)  # show all columns, don't truncate

In [2]:
path = "../data/raw/"

customers = pd.read_csv(path + "olist_customers_dataset.csv")
orders = pd.read_csv(path + "olist_orders_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")

In [3]:
# Summary of all datasets
print("DATASET SUMMARY")
print("=" * 50)
for name, df in zip(['customers', 'orders', 'order_items', 'payments', 'reviews', 'products', 'sellers', 'geolocation', 'category_translation'], 
                    [customers, orders, order_items, payments, reviews, products, sellers, geolocation, category_translation]):
    print(f"{name:20} | {df.shape[0]:>8} rows | {df.shape[1]:>3} columns")

DATASET SUMMARY
customers            |    99441 rows |   5 columns
orders               |    99441 rows |   8 columns
order_items          |   112650 rows |   7 columns
payments             |   103886 rows |   5 columns
reviews              |    99224 rows |   7 columns
products             |    32951 rows |   9 columns
sellers              |     3095 rows |   4 columns
geolocation          |  1000163 rows |   5 columns
category_translation |       71 rows |   2 columns


In [4]:
def inspect(df, name):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("\nDtypes:\n", df.dtypes)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nHead:")
    display(df.head(3))
    print("\n")

In [5]:
inspect(orders, "orders")

--- orders ---
Shape: (99441, 8)

Dtypes:
 order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing values:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate rows: 0

Head:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [14]:
date_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [15]:
# Check if conversion worked
print(orders[['order_purchase_timestamp', 'order_delivered_customer_date']].head())

# Check missing values after conversion
print(orders[date_cols].isna().sum())

  order_purchase_timestamp order_delivered_customer_date
0      2017-10-02 10:56:33           2017-10-10 21:25:13
1      2018-07-24 20:41:37           2018-08-07 15:27:45
2      2018-08-08 08:38:49           2018-08-17 18:06:29
3      2017-11-18 19:28:06           2017-12-02 00:28:42
4      2018-02-13 21:18:39           2018-02-16 18:17:02
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [6]:
inspect(customers, "customers")

--- customers ---
Shape: (99441, 5)

Dtypes:
 customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing values:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows: 0

Head:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [16]:
# Unique customers vs unique orders
print(f"Unique customer IDs: {customers['customer_id'].nunique()}")
print(f"Unique customer unique IDs: {customers['customer_unique_id'].nunique()}")
print(f"Total orders: {len(orders)}")

Unique customer IDs: 99441
Unique customer unique IDs: 96096
Total orders: 99441


In [17]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str)

In [7]:
inspect(order_items, "order_items")

--- order_items ---
Shape: (112650, 7)

Dtypes:
 order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Missing values:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Duplicate rows: 0

Head:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [18]:
# Calculate average items per order
avg_items = len(order_items) / len(orders)
print(f"Average items per order: {avg_items:.2f}")
# Expected: ~1.13

Average items per order: 1.13


In [19]:
# Convert shipping date
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

# Verify
print(order_items['shipping_limit_date'].dtype)


datetime64[us]


In [8]:
inspect(payments, "payments")

--- payments ---
Shape: (103886, 5)

Dtypes:
 order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Missing values:
 order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Duplicate rows: 0

Head:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [9]:
inspect(reviews, "reviews")

--- reviews ---
Shape: (99224, 7)

Dtypes:
 review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

Missing values:
 review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Duplicate rows: 0

Head:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [20]:
# Convert date columns
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# Verify
print(reviews[['review_creation_date', 'review_answer_timestamp']].dtypes)

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [10]:
inspect(products, "products")

--- products ---
Shape: (32951, 9)

Dtypes:
 product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

Missing values:
 product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Duplicate rows: 0

Head:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


In [21]:
products = products.merge(category_translation, on='product_category_name', how='left')
products[['product_category_name', 'product_category_name_english']].head()

,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


In [22]:
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

In [11]:
inspect(sellers, "sellers")

--- sellers ---
Shape: (3095, 4)

Dtypes:
 seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

Missing values:
 seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Duplicate rows: 0

Head:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


In [23]:
# Convert ZIP to string (should already be done)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

# Verify
print(sellers['seller_zip_code_prefix'].dtype)
print(sellers['seller_zip_code_prefix'].head())

str
0    13023
1    13844
2    20031
3    04195
4    12914
Name: seller_zip_code_prefix, dtype: str


In [12]:
inspect(geolocation, "geolocation")

--- geolocation ---
Shape: (1000163, 5)

Dtypes:
 geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Missing values:
 geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicate rows: 261831

Head:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


In [24]:
# Convert geolocation ZIP to string
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5)

In [13]:
inspect(category_translation, "category_translation")

--- category_translation ---
Shape: (71, 2)

Dtypes:
 product_category_name            str
product_category_name_english    str
dtype: object

Missing values:
 product_category_name            0
product_category_name_english    0
dtype: int64

Duplicate rows: 0

Head:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [25]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [26]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
orders_delivered.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [29]:
# Drop only rows where delivery_customer_date is missing

orders_delivered = orders_delivered.dropna(subset=['order_delivered_customer_date'])
orders_delivered.isnull().sum()


order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
dtype: int64

In [27]:
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                  ('payments', payments), ('reviews', reviews), ('products', products),
                  ('sellers', sellers)]:
    print(name, df.duplicated().sum())

orders 0
customers 0
order_items 0
payments 0
reviews 0
products 0
sellers 0


In [28]:
print((order_items['price'] < 0).sum())
print((order_items['freight_value'] < 0).sum())
print((payments['payment_value'] < 0).sum())

0
0
0


In [30]:
orders.to_csv('../data/processed/orders_clean.csv', index=False)
orders_delivered.to_csv('../data/processed/orders_delivered.csv', index=False)
products.to_csv('../data/processed/products_clean.csv', index=False)
customers.to_csv('../data/processed/customers_clean.csv', index=False)
order_items.to_csv('../data/processed/order_items_clean.csv', index=False)
payments.to_csv('../data/processed/payments_clean.csv', index=False)
reviews.to_csv('../data/processed/reviews_clean.csv', index=False)
sellers.to_csv('../data/processed/sellers_clean.csv', index=False)

In [32]:
# Check that dates are still datetime
orders_loaded = pd.read_csv('../data/processed/orders_clean.csv')

# Check date columns
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

print("Date columns in loaded file:")
for col in date_cols:
    print(f"  {col}: {orders_loaded[col].dtype}")

Date columns in loaded file:
  order_purchase_timestamp: str
  order_approved_at: str
  order_delivered_carrier_date: str
  order_delivered_customer_date: str
  order_estimated_delivery_date: str
